In [ ]:
# !pip install requests beautifulsoup4 lxml -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import requests
from bs4 import BeautifulSoup
import urllib3
import os
import json
from urllib.parse import urljoin
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock
import time
from datetime import datetime


In [ ]:
GG_COLAB = "/content/drive/MyDrive/"
LABORLAW_DIR = f'{GG_COLAB}/laborlaw/'
SAVE_PDF_DIR = f'{GG_COLAB}/laborlaw/downloaded_pdf'
URL_FILE = f'{GG_COLAB}/laborlaw/collected_urls/ban_an_lao_dong.json'
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [ ]:
class CourtPDFDownloader:
    def __init__(self, output_dir=SAVE_PDF_DIR, max_workers=10):
        self.base_url = "https://congbobanan.toaan.gov.vn"
        self.output_dir = output_dir
        self.max_workers = max_workers  # Số luồng song song
        self.lock = Lock()
        # File để lưu lỗi
        self.error_log_file = f'{LABORLAW_DIR}/errors.log'
        self.progress_file =f'{LABORLAW_DIR}/download_progress.txt'
        # Headers chung
        self.headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
            'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
            'Accept-Language': 'vi-VN,vi;q=0.9,en;q=0.8',
        }
        os.makedirs(self.output_dir, exist_ok=True)
    def save_progress(self, filename):
        #Lưu tên file đã tải thành công
        with open(self.progress_file, 'a', encoding='utf-8') as f:
            f.write(f"{filename}\n")

    def load_progress(self):
        #Đọc list file đã tải
        if os.path.exists(self.progress_file):
            with open(self.progress_file, 'r', encoding='utf-8') as f:
                return set(line.strip() for line in f)
        return set()
    def sanitize_filename(self, filename):
        # Loại bỏ ký tự không hợp lệ trong tên file
        filename = filename.replace('/', '').replace('\\', '')
        filename = "".join(c for c in filename if c.isalnum() or c in (' ', '.', '_', '-')).strip()

        if not filename.endswith('.pdf'):
            filename += '.pdf'

        return filename if filename else 'file.pdf'

    # ===== THÊM MỚI: Hàm ghi log lỗi =====
    def log_error(self, message):
        #Ghi lỗi vào file log
        try:
            with open(self.error_log_file, 'a', encoding='utf-8') as f:
                timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
                f.write(f"[{timestamp}] {message}\n")
        except Exception as e:
            print(f"Không thể ghi log: {e}")

    def download_single_pdf(self, pdf_url, save_as,max_retries=3):
        # tải 1 file pdf
        #Thêm max_retries=3: Tự động retry 3 lần
        #Exponential backoff: 1s, 2s, 4s
        #Lưu progress sau khi thành công
        filepath = os.path.join(self.output_dir, save_as)
        # Bỏ qua nếu file đã tồn tại
        if os.path.exists(filepath):
            with self.lock:
                print(f"-------> Đã có: {save_as}")
            return True
        # Retry loop
        for attempt in range(max_retries):
            try:
                # Tạo session riêng cho mỗi thread
                session = requests.Session()
                session.headers.update(self.headers)

                response = session.get(pdf_url, verify=False, timeout=(60, 120), stream=True) # (connect timeout, read timeout)

                if response.status_code != 200:
                    with self.lock:
                        error_msg = f"====>Lỗi HTTP {response.status_code}: {save_as}"
                        print(error_msg)
                        self.log_error(error_msg)
                    return False

                filepath = os.path.join(self.output_dir, save_as)

                with open(filepath, 'wb') as f:
                    for chunk in response.iter_content(chunk_size=8192):
                        if chunk:
                            f.write(chunk)

                file_size = os.path.getsize(filepath)
                self.save_progress(save_as)
                with self.lock:
                    print(f"====>Đã lưu: {save_as} ({file_size:,} bytes)")

                return True

            except Exception as e:
                if attempt < max_retries - 1:
                    #Retry với backoff
                    wait_time = 2 ** attempt  # 1s, 2s, 4s
                    with self.lock:
                        print(f"Lỗi {save_as} (lần {attempt+1}/{max_retries}), retry sau {wait_time}s... [{e}]")
                    time.sleep(wait_time)
                    continue
                else:
                    error_msg=f"====>Fail {save_as}: {e}"
                    with self.lock:
                        print(error_msg)
                        self.log_error(error_msg)
                    return False

    def download_from_url(self, detail_url, prefix_number=None):
        #Tải PDF từ 1 URL chi tiết
        try:
            # Tạo session riêng
            session = requests.Session()
            session.headers.update(self.headers)

            response = session.get(detail_url, verify=False, timeout=(60, 120))

            if response.status_code != 200:
                with self.lock:
                    error_msg = f"=>error:[{prefix_number}] Lỗi HTTP {response.status_code}"
                    print(error_msg)
                    self.log_error(error_msg)
                return []

        except Exception as e:
            with self.lock:
                error_msg = f"=>error:[{prefix_number}] Lỗi: {e}"
                print(error_msg)
                self.log_error(error_msg)
            return []

        # Parse HTML tìm link PDF
        soup = BeautifulSoup(response.text, 'lxml')
        pdf_links = []

        for a in soup.find_all('a', href=True):
            href = a['href']
            has_download = a.has_attr('download')

            if has_download or '.pdf' in href.lower():
                full_url = urljoin(self.base_url, href)
                filename = a.get_text(strip=True)
                pdf_links.append({'url': full_url, 'filename': filename})

        if not pdf_links:
            with self.lock:
                print(f"====>[{prefix_number}] Không tìm thấy PDF")
            return []

        with self.lock:
            print(f"--> [{prefix_number}] Tìm thấy {len(pdf_links)} file PDF")

        # Chuẩn bị list file cần tải
        download_tasks = []
        for i, pdf in enumerate(pdf_links, 1):
            clean_name = self.sanitize_filename(pdf['filename'])

            if prefix_number is not None:
                if len(pdf_links) > 1:
                    # save_as = f"{prefix_number}_{i}_{clean_name}"
                    save_as = f"{prefix_number}_{i}.pdf"  # 5_1.pdf, 5_2.pdf
                else:
                    # save_as = f"{prefix_number}_{clean_name}"
                    save_as = f"{prefix_number}.pdf"      # 5.pdf
            else:
                # save_as = clean_name
                save_as = f"{i}.pdf"  # 1.pdf, 2.pdf, 3.pdf

            download_tasks.append((pdf['url'], save_as))

        return download_tasks



In [ ]:
def download_from_json(json_file, output_dir=SAVE_PDF_DIR, max_workers=10):
    """
    json_file: path file JSON
    output_dir: Thư mục lưu file
    max_workers: Số luồng song song
    """
    # Đọc file JSON
    with open(json_file, 'r', encoding='utf-8') as f:
        data = json.load(f)
    print(f'Tải {len(data)} url')
    print(f'song song {max_workers} luồng')


    downloader = CourtPDFDownloader(output_dir=output_dir, max_workers=max_workers)
    #Đọc progress
    downloaded_files = downloader.load_progress()
    print(f'=> Đã tải trước đó {len(downloaded_files)} file')
    start_time = time.time()

    # bước 1: thu thập all link PDF từ các trang chi tiết (song song)

    all_pdf_tasks = []

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_index = {
            executor.submit(downloader.download_from_url, item['url'], i): i
            for i, item in enumerate(data, 1)
        }

        for future in as_completed(future_to_index):
            tasks = future.result()
            all_pdf_tasks.extend(tasks)

    print(f"\n=>Tổng cộng: {len(all_pdf_tasks)} file PDF cần tải\n")

    # bước 2: Tải all PDF (song song)
    # Lọc file đã tải
    original_count = len(all_pdf_tasks)
    all_pdf_tasks = [
        (url, filename) for url, filename in all_pdf_tasks
        if filename not in downloaded_files
    ]
    skipped = original_count - len(all_pdf_tasks)
    if skipped > 0:
        print(f"----> Bỏ qua {skipped} file đã tải, còn {len(all_pdf_tasks)} file")
    if len(all_pdf_tasks) == 0:
        print("Tất cả đã được tải!")
        return
    # Tải PDF với retry
    print(f'Sau khi lọc file đã tải ---> Tải {len(all_pdf_tasks)} file')
    downloaded_count = 0
    failed_count = 0
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {
            executor.submit(downloader.download_single_pdf, url, filename): filename
            for url, filename in all_pdf_tasks
        }

        for future in as_completed(futures):
            if future.result():
                downloaded_count += 1
            else:
                failed_count += 1

    elapsed = time.time() - start_time


    print(f"====>Success: {downloaded_count}/{len(all_pdf_tasks)} file")
    print(f"====>Fail: {failed_count}/{len(all_pdf_tasks)} file")
    print(f"====> Đã tải: {downloaded_count}/{len(all_pdf_tasks)} file")
    print(f"====> Thời gian: {elapsed:.1f} giây ({elapsed/60:.1f} phút)")
    print(f"====> Tốc độ: {downloaded_count/(elapsed/60):.1f} file/phút")


In [ ]:
download_from_json(
    json_file=URL_FILE,
    output_dir= SAVE_PDF_DIR,
    max_workers=50
)

Kết quả truyền trực tuyến bị cắt bớt đến 5000 dòng cuối.
--> [2793] Tìm thấy 1 file PDF
--> [2791] Tìm thấy 1 file PDF
--> [2800] Tìm thấy 1 file PDF
--> [2798] Tìm thấy 1 file PDF
=>error:[2168] Lỗi: HTTPSConnectionPool(host='congbobanan.toaan.gov.vn', port=443): Read timed out. (read timeout=120)
--> [2792] Tìm thấy 1 file PDF
--> [2805] Tìm thấy 1 file PDF
--> [2762] Tìm thấy 1 file PDF
--> [2734] Tìm thấy 1 file PDF
--> [2787] Tìm thấy 1 file PDF
--> [2808] Tìm thấy 1 file PDF
--> [2797] Tìm thấy 1 file PDF
--> [2811] Tìm thấy 1 file PDF
--> [2803] Tìm thấy 1 file PDF
--> [2813] Tìm thấy 1 file PDF
--> [2801] Tìm thấy 1 file PDF
--> [2696] Tìm thấy 1 file PDF
--> [2774] Tìm thấy 1 file PDF
--> [2790] Tìm thấy 1 file PDF
--> [2806] Tìm thấy 1 file PDF
--> [2739] Tìm thấy 1 file PDF
--> [2826] Tìm thấy 1 file PDF
--> [2727] Tìm thấy 1 file PDF
--> [2807] Tìm thấy 1 file PDF
--> [2825] Tìm thấy 1 file PDF
--> [2700] Tìm thấy 1 file PDF
--> [2815] Tìm thấy 1 file PDF
--> [2736] Tìm thấ